In [1]:
import numpy as np
import pandas as pd
import os
import torch

from sklearn.metrics import cohen_kappa_score

In [2]:
SCORE_RANGES = {
        1: {'sentence_fluency': (1, 6), 'word_choice': (1, 6), 'conventions': (1, 6),'organization': (1, 6),
            'content': (1, 6), 'holistic': (2, 12)},
        2: {'sentence_fluency': (1, 6), 'word_choice': (1, 6), 'conventions': (1, 6),'organization': (1, 6),
            'content': (1, 6), 'holistic': (1, 6)},
        3: {'narrativity': (0, 3), 'language': (0, 3), 'prompt_adherence': (0, 3), 'content': (0, 3), 'holistic': (0, 3)},
        4: {'narrativity': (0, 3), 'language': (0, 3), 'prompt_adherence': (0, 3), 'content': (0, 3), 'holistic': (0, 3)},
        5: {'narrativity': (0, 4), 'language': (0, 4), 'prompt_adherence': (0, 4), 'content': (0, 4), 'holistic': (0, 4)},
        6: {'narrativity': (0, 4), 'language': (0, 4), 'prompt_adherence': (0, 4), 'content': (0, 4), 'holistic': (0, 4)},
        7: {'conventions': (0, 6), 'organization': (0, 6), 'content': (0, 6),'holistic': (0, 30)},
        8: {'sentence_fluency': (2, 12), 'word_choice': (2, 12), 'conventions': (2, 12),'organization': (2, 12),
            'content': (2, 12), 'holistic': (0, 60)}}

In [3]:
def scaleUpTrait(Data, SCORE_RANGES, prompt,score):
    min_score, max_score = SCORE_RANGES[prompt][score]
    scalledUpData = (Data * (max_score-min_score) ) + min_score
    return scalledUpData

In [4]:
# a function to check if a score exists within a prompt id
def score_exists(prompt_id, trait, score):
    prompt = SCORE_RANGES.get(prompt_id)  #get the prompt
    if prompt is None:
        return False
    
    traits = prompt.get(trait)  #get the trait
    if traits is None:
        return False  
    
    # checking if the score is within the range
    return SCORE_RANGES[prompt_id][trait][0] <= score <= SCORE_RANGES[prompt_id][trait][1]

In [38]:
def scaleUpTraits(prompt_id, y_values,SCORE_RANGES):
    TraitS = ['holistic','content','organization','word_choice','sentence_fluency','conventions','prompt_adherence', 'language', 'narrativity']
    newY = y_values
    i = -1
    for trait in TraitS:
        i = i+1
        if score_exists(prompt_id, trait, 2):
            minScore, maxScore = SCORE_RANGES[prompt_id][trait]
            scaled_value = y_values[i]*(maxScore-minScore)+minScore  # Scale the value
            newY[i] = np.round(scaled_value) 
        else:
            newY[i] = 0
    return newY

In [27]:
def predictScores(features, prompt_id, approach, SCORE_RANGES, votingModel):
    
    featuresTensor = torch.tensor(features.values, dtype=torch.float32)

    #predict scores
    output = votingModel(featuresTensor)
    output = output.detach().cpu().numpy()

    #scale the output
    if approach =="A":
        #scale holistic score
        output = scaleUpTrait(output, SCORE_RANGES, prompt_id, 'holistic')
    elif approach =="B":
        #scale nine scores includes traits and holistic
        output = scaleUpTraits(prompt_id,output,SCORE_RANGES)

    #return rounded predictions as integers
    return np.round(output).astype(int)

In [39]:
def AESsystem():
   
    neededColumns = ['mean_word', 'word_var', 'mean_sent', 'sent_var', 'ess_char_len', 'word_count', 'prep_comma',
                     'unique_word', 'clause_per_s', 'mean_clause_l', 'max_clause_in_s', 'spelling_err',
                     'sent_ave_depth', 'ave_leaf_depth', 'automated_readability', 'linsear_write', 'stop_prop',
                     'positive_sentence_prop', 'negative_sentence_prop', 'neutral_sentence_prop',
                     'overall_positivity_score', 'overall_negativity_score', ',', '.', 'VB', 'JJR', 'WP', 'PRP$', 'VBN',
                     'VBG', 'IN', 'CC', 'JJS', 'PRP', 'MD', 'WRB', 'RB', 'VBD', 'RBR', 'VBZ', 'NNP', 'POS', 'WDT', 'DT',
                     'CD', 'NN', 'TO', 'JJ', 'VBP', 'RP', 'NNS', 'Kincaid', 'ARI', 'Coleman-Liau', 'FleschReadingEase',
                     'GunningFogIndex', 'LIX', 'SMOGIndex', 'RIX', 'DaleChallIndex', 'characters_per_word',
                     'syll_per_word', 'words_per_sentence', 'sentences_per_paragraph', 'type_token_ratio',
                     'characters', 'syllables', 'words', 'wordtypes', 'sentences', 'paragraphs', 'long_words',
                     'complex_words', 'complex_words_dc', 'tobeverb', 'auxverb', 'conjunction', 'pronoun',
                     'preposition', 'nominalization', 'begin_w_pronoun', 'begin_w_interrogative', 'begin_w_article',
                     'begin_w_subordination', 'begin_w_conjunction', 'begin_w_preposition']

    print(''' 
     Welcome to the Deep-Learning-based AES system.
     This system predicts scores based on the chosen approach.
     Approach A: Predicts holistic scores.
     Approach B: Predicts holistic and trait scores.
    ''')
    approach=input("Type A or B").strip().upper()
    if approach not in ["A","B"]:
        print("Invalid approach. Please restart the system and choose 'A' or 'B'.")
        return

    while True:
        try:
            path = input("(type 'exit' to exit) Paste the csv file path: ")

            if path =="exit":
                print("Thank you for using the AES system!")
                break

            if not os.path.exists(path):
                raise FileNotFoundError("The file does not exist. Please try again.")

            inputData = pd.read_csv(path)
            print("File successfully read...")

            if 'essay_id' not in inputData.columns or 'prompt_id' not in inputData.columns or 'mean_word' not in inputData.columns:
                raise ValueError("The CSV file must contain 'essay_id', 'prompt_id', and 'mean_word' columns.")

            mean_word_index = inputData.columns.get_loc('mean_word')
            if mean_word_index + 86 >len(inputData.columns):
                raise ValueError("The file does not have the required 86 feature columns after 'mean_word'.")

            providedColumns = inputData.columns[mean_word_index:mean_word_index + 86]
            if list(providedColumns)!=neededColumns:
                raise ValueError("The columns after 'mean_word' do not match the required feature names and order.")

            print("CSV file meets the requirements...")

            #select path according to approach
            filePath = f"model-{approach}-deploy.pt"
            if not os.path.exists(filePath):
                raise FileNotFoundError(f"Deployed model file for Approach {approach} not found!")
        
            #load deployed model
            votingModel = torch.jit.load(filePath)

            #Defininng column names based on the approach
            if approach == "A":
                 newColumns = ['holistic']
            elif approach == "B":
                newColumns = ['holistic', 'content', 'organization', 'word_choice','sentence_fluency', 'conventions', 'prompt_adherence', 'language', 'narrativity']
                    
            groupedData = inputData.groupby('prompt_id')
            result_list = []

            for prompt_id, group in groupedData:
                group_predictions = []  #to store predictions for the group
            
                for column, row in group.iterrows():
                    #Extract features for the current row
                    features = row[neededColumns]
            
                    #Call predictScores with a single row of features
                    predictions = predictScores(features, prompt_id, approach, SCORE_RANGES, votingModel)
            
                    #Append predictions to the list
                    group_predictions.append(predictions)
            
                #Convert predictions into a DataFrame
                predictions_df = pd.DataFrame(group_predictions, columns=newColumns, index=group.index)
            
                #Concatenate predictions with the old group
                group = pd.concat([group, predictions_df], axis=1)
            
                #Add the group to the result list
                result_list.append(group[['essay_id', 'prompt_id'] + newColumns + neededColumns])
            
            #Concatenate all the groups
            final_result = pd.concat(result_list)

            output_path = f"Scored Essays (Approach {approach}).csv"
            final_result.to_csv(output_path, index=False)
            print(f"Data saved to {output_path}!")
            break

        except FileNotFoundError as fnf_error:
            print(fnf_error)
        except pd.errors.EmptyDataError:
            print("The file is empty. Please try again.")
        except pd.errors.ParserError:
            print("The file could not be parsed as as CSV. Please check the file format and try again.")
        except ValueError as val_error:
            print(val_error)
        except Exception as e:
            print(f"An unexpected error occurred: {e}. Please try again.")



In [41]:
AESsystem()

 
     Welcome to the Deep-Learning-based AES system.
     This system predicts scores based on the chosen approach.
     Approach A: Predicts holistic scores.
     Approach B: Predicts holistic and trait scores.
    


Type A or B A
(type 'exit' to exit) Paste the csv file path:  Essays to score.csv


File successfully read...
CSV file meets the requirements...
Data saved to Scored Essays (Approach A).csv!
